In [2]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

roll_number = "1024160009"
last_two_digits = roll_number[-2:]

categories = ["billing", "account", "general"]

personalized_entries = []

for digit in last_two_digits:
    d = int(digit)
    category = categories[d % 3]

    if category == "billing":
        personalized_entries.append({
            "question": "how can i check my payment status",
            "answer": "You can check your payment status from the billing section.",
            "keywords": "payment status billing",
            "category": category
        })
    elif category == "account":
        personalized_entries.append({
            "question": "how do i update my registered mobile number",
            "answer": "Go to Settings > Account and update your registered mobile number.",
            "keywords": "mobile number update",
            "category": category
        })
    else:
        personalized_entries.append({
            "question": "where can i find general information",
            "answer": "You can find general information in the help section.",
            "keywords": "information help details",
            "category": category
        })

df = pd.DataFrame(fixed_entries + personalized_entries)

print(df)

                            question  \
0             what is the annual fee   
1              how to reset password   
2        what are your working hours   
3              how can i pay the fee   
4  how can i check my payment status   
5  how can i check my payment status   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4  You can check your payment status from the bil...  payment status billing   
5  You can check your payment status from the bil...  payment status billing   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  billing  


In [3]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        score = len(query_words & keywords) + len(query_words & question_words)

        if score > 0:
            results.append((score, row["question"], row["answer"], row["category"]))

    results.sort(reverse=True)

    for score, question, answer, category in results:
        print("Score:", score)
        print("Question:", question)
        print("Answer:", answer)
        print("Category:", category)
        print()

score_query("fee payment", df)

Score: 3
Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing

Score: 2
Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing

Score: 2
Question: how can i check my payment status
Answer: You can check your payment status from the billing section.
Category: billing

Score: 2
Question: how can i check my payment status
Answer: You can check your payment status from the billing section.
Category: billing



In [4]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

result = same_category("billing", df)

print(result)

0               what is the annual fee
3                how can i pay the fee
4    how can i check my payment status
5    how can i check my payment status
Name: question, dtype: object


In [5]:
entry_index = 0

new_keyword = input("Enter a new keyword: ")

df.loc[entry_index, "keywords"] = df.loc[entry_index, "keywords"] + " " + new_keyword

df.to_csv("1024160009_faq_data.csv", index=False)

print(df)

Enter a new keyword: annual
                            question  \
0             what is the annual fee   
1              how to reset password   
2        what are your working hours   
3              how can i pay the fee   
4  how can i check my payment status   
5  how can i check my payment status   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can check your payment status from the bil...   
5  You can check your payment status from the bil...   

                       keywords category  
0  fee cost price charge annual  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4        payment status billing  billing  
5        payment status billing  billing

In [6]:
print(df.groupby("category")["question"].count())

category
account    1
billing    4
general    1
Name: question, dtype: int64


In [7]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())

        score = len(query_words & keywords) + len(query_words & question_words)

        if score > 0:
            results.append((score, row))

    if not results:
        print("No matching entries found.")
        return

    max_score = max(score for score, row in results)

    top_matches = [(score, row) for score, row in results if score == max_score]

    print("Highest Score:", max_score)

    for score, row in top_matches:
        print("Question:", row["question"])
        print("Answer:", row["answer"])
        print("Category:", row["category"])
        print()

print("Tie Query:")
score_query_with_ties("fee", df)

print("Non-Tie Query:")
score_query_with_ties("password", df)

Tie Query:
Highest Score: 2
Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing

Non-Tie Query:
Highest Score: 2
Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account

